# 07 · Experimental ML land cover
Predict NLCD-style cover using NAIP, seasonal Sentinel-2 and terrain. Labels are weak 30 m NLCD labels, not field truth. Goat Island is excluded from training. Geographic holdouts measure agreement with NLCD; NWI is an external comparison, not ground truth.

Run sequentially in local Jupyter inside the repository. Earth Engine requires internet and an authenticated account. No existing environmental outputs are overwritten. Output: `goat_island_outputs/ml_land_cover/`.

Dependencies: earthengine-api, geemap, geopandas, shapely, rasterio, numpy, pandas, matplotlib, scikit-learn, joblib. Install in the notebook kernel environment if missing; this notebook does not automatically install packages.


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, warnings
import ee, geemap, geopandas as gpd
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import geometry_mask
from rasterio.plot import show
from shapely import force_2d
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, f1_score
import joblib
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from IPython.display import display


## Settings and scope
Defaults use broader Texas training areas, Texas holdouts and a Missouri transfer holdout. This remains a regional experiment, not nationwide training. Some regions contain very unequal target-class availability; inspect region-level results before interpreting transfer performance. Region coordinates are candidate sampling boxes, not verified wetland reference sites; the availability and class-count checks below decide whether they are usable. Add well-separated regions to expand the experiment.

The 2022 NAIP year matches the project's existing aerial reference. Confidence thresholds are candidate filters, not calibrated accuracy guarantees. Values outside 1–100 are rejected. Check the source release metadata and USGS confidence caveats before interpreting results.


In [ ]:
from pathlib import Path

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "Park_Parcels/Park_Parcles.shp").exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open Jupyter inside goat-island-analysis.")
OUT = PROJECT_ROOT / "goat_island_outputs/ml_land_cover"
OUT.mkdir(parents=True, exist_ok=True)
EE_PROJECT = "stalwart-glider-279103"
YEAR = 2022
CONFIDENCE_THRESHOLDS = [80, 90]
SAMPLES_PER_CLASS_REGION = 120
# Increase only training forest samples; retain existing holdout sampling.
TRAIN_CLASS_QUOTAS = {41: 240}
MIN_TRAIN_PER_CLASS = 20
SEED = 6379
OUTPUT_SCALE_M = 2
METRIC_CRS = "EPSG:26914"
PARCEL_IDS = ["65079661510030000", "65130905010020000", "65130905010030000", "65130905010050000", "65130905010060000", "65146334010100000", "65146334010140000"]
CLASSES = {11: "Open water", 21: "Developed open space", 22: "Developed low intensity", 23: "Developed medium intensity", 24: "Developed high intensity", 31: "Barren", 41: "Deciduous forest", 42: "Evergreen forest", 43: "Mixed forest", 52: "Shrub", 71: "Herbaceous", 81: "Pasture/hay", 82: "Crops", 90: "Woody wetland", 95: "Herbaceous wetland"}
REGIONS = [('north_texas', 'train', [-97.1, 33.0, -96.6, 33.5]), ('ne_texas', 'train', [-95.6, 33.1, -95.1, 33.6]), ('north_texas_holdout', 'test', [-97.6, 33.4, -97.1, 33.9]), ('red_river_holdout', 'test', [-95.0, 33.0, -94.5, 33.5]), ('missouri', 'test', [-91.5, 36.5, -91.0, 37.0])]

# Optional polygon file with numerical fields hydric_pct (0–100) and drainage_code.
# Prepare from SSURGO component-weighted map-unit attributes; do not assign components to precise subareas.
SOIL_PATH = None
# Optional downloaded NWI polygon file. It NEVER enters the feature stack or training labels.
NWI_PATH = None


In [ ]:
try:
    ee.Initialize(project=EE_PROJECT)
except ee.EEException:
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT)
parcels = gpd.read_file(PROJECT_ROOT / "Park_Parcels/Park_Parcles.shp")
if parcels.crs is None:
    raise ValueError("Parcel CRS missing")
selected = parcels.loc[parcels.Parcel_Num.astype(str).str.strip().isin(PARCEL_IDS)].copy()
if set(selected.Parcel_Num.astype(str).str.strip()) != set(PARCEL_IDS):
    raise ValueError("Seven parcel IDs not found")
selected.geometry = selected.geometry.map(force_2d).make_valid()
preserve = selected.dissolve().to_crs(4326)
preserve_m = preserve.to_crs(METRIC_CRS)
roi = ee.Geometry(json.loads(json.dumps(preserve.geometry.iloc[0].__geo_interface__)))
exclusion = roi.buffer(10000)
BOUNDARY_HASH = hashlib.sha256(preserve.geometry.iloc[0].wkb).hexdigest()

import requests
import math
import xml.etree.ElementTree as ET
from rasterio.warp import transform_bounds

NLCD_SERVICE_RELEASE = "Collection 1.2"
NLCD_SERVICE_ROOT = "https://dmsdata.cr.usgs.gov/geoserver"

def official_nlcd_pair(bounds, region_id):
    """Fetch raw, nearest-neighbor 2022 labels/confidence on the same native 30 m grid."""
    cache = OUT / "official_nlcd" / "collection_1_2" / str(YEAR) / region_id
    cache.mkdir(parents=True, exist_ok=True)
    b = transform_bounds("EPSG:4326", "EPSG:5070", *bounds)
    left = math.floor((b[0] + 2415585) / 30)*30 - 2415585
    bottom = math.floor((b[1] - 164805) / 30)*30 + 164805
    width, height = math.ceil((b[2]-left)/30), math.ceil((b[3]-bottom)/30)
    result = []
    records = {}
    for product in ["Land-Cover", "Land-Cover-Confidence"]:
        layer = f"{product}-Native_conus_year_data"
        workspace = f"mrlc_{layer}"
        url = f"{NLCD_SERVICE_ROOT}/{workspace}/wcs"
        params = {"service":"WCS", "request":"GetCoverage", "version":"1.0.0", "coverage":f"{workspace}:{layer}", "crs":"EPSG:5070", "response_crs":"EPSG:5070", "bbox":f"{left},{bottom},{left+width*30},{bottom+height*30}", "width":width, "height":height, "format":"GeoTIFF", "time":f"{YEAR}-01-01T00:00:00.000Z", "interpolation":"nearest neighbor"}
        path = cache / f"{product}.tif"
        receipt = cache / f"{product}.json"
        request_signature = hashlib.sha256(json.dumps(params,sort_keys=True).encode()).hexdigest()
        record = json.loads(receipt.read_text()) if receipt.exists() else {}
        if not path.exists() or record.get("request_signature") != request_signature or record.get("sha256") != hashlib.sha256(path.read_bytes()).hexdigest():
            response = requests.get(url, params=params, timeout=180)
            response.raise_for_status()
            if "tiff" not in response.headers.get("Content-Type", "").lower():
                raise RuntimeError(f"USGS returned no GeoTIFF: {response.text[:500]}")
            partial = path.with_suffix(".partial.tif")
            partial.write_bytes(response.content)
            with rasterio.open(partial) as src:
                if src.count != 1 or src.shape != (height,width) or abs(src.transform.a-30) > 0.001:
                    raise ValueError("USGS raster dimensions/resolution differ from request")
            partial.replace(path)
            record = {"url":response.url,"release":NLCD_SERVICE_RELEASE,"year":YEAR,"request_signature":request_signature,"sha256":hashlib.sha256(path.read_bytes()).hexdigest(),"retrieved_utc":datetime.now(timezone.utc).isoformat(),"params":params}
            receipt.write_text(json.dumps(record,indent=2))
        with rasterio.open(path) as src:
            result.append((src.read(1),src.transform,src.crs))
        records[product] = record
    if result[0][0].shape != result[1][0].shape or not result[0][1].almost_equals(result[1][1]) or result[0][2] != result[1][2]:
        raise ValueError("USGS label/confidence grids differ")
    return result[0][0], result[1][0], result[0][1], records

def official_training_points(bounds, region_id):
    land, conf, transform, records = official_nlcd_pair(bounds, region_id)
    neighborhoods = np.lib.stride_tricks.sliding_window_view(land, (3,3))
    interior = np.zeros(land.shape, dtype=bool)
    interior[1:-1,1:-1] = neighborhoods.min(axis=(-2,-1)) == neighborhoods.max(axis=(-2,-1))
    # Restrict the projected bounding rectangle to the original geographic region
    # and exclude the study parcels plus 10 km before sampling.
    from shapely.geometry import box
    polygon = gpd.GeoSeries([box(*bounds)],crs=4326).to_crs(5070).iloc[0]
    excluded = preserve_m.buffer(10000).to_crs(5070).geometry.iloc[0]
    region_mask = geometry_mask([polygon.difference(excluded).__geo_interface__],out_shape=land.shape,transform=transform,invert=True)
    eligible = interior & region_mask & (conf >= min(CONFIDENCE_THRESHOLDS)) & (conf <= 100)
    rng = np.random.default_rng(SEED)
    features, counts = [], []
    for code in CLASSES:
        for threshold in CONFIDENCE_THRESHOLDS:
            counts.append({"region":region_id,"class":code,"threshold":threshold,"eligible_cells":int((interior & region_mask & (land==code) & (conf>=threshold) & (conf<=100)).sum())})
        # Separate confidence bins preserve high-confidence examples for the 90 experiment.
        for low,high in [(80,90),(90,101)]:
            indices = np.flatnonzero(eligible & (land==code) & (conf>=low) & (conf<high))
            if len(indices) == 0:continue
            candidates = indices
            # Keep the original RNG draw so other classes and holdout rows stay stable.
            indices = rng.choice(candidates,size=min(SAMPLES_PER_CLASS_REGION,len(candidates)),replace=False)
            is_training_region = any(name == region_id and split == "train" for name,split,_ in REGIONS)
            quota = TRAIN_CLASS_QUOTAS.get(code,SAMPLES_PER_CLASS_REGION) if is_training_region else SAMPLES_PER_CLASS_REGION
            if quota > len(indices):
                remaining = np.setdiff1d(candidates,indices)
                extra_seed = int(hashlib.sha256(f"{SEED}:{region_id}:{code}:{low}:extra".encode()).hexdigest()[:8],16)
                extra_rng = np.random.default_rng(extra_seed)
                extra = extra_rng.choice(remaining,size=min(quota-len(indices),len(remaining)),replace=False)
                indices = np.concatenate([indices,extra])
            rows,cols = np.unravel_index(indices,land.shape)
            x,y = rasterio.transform.xy(transform,rows,cols)
            points = gpd.GeoSeries(gpd.points_from_xy(x,y),crs=5070).to_crs(4326)
            for flat,point in zip(indices,points):
                features.append(ee.Feature(ee.Geometry.Point([point.x,point.y]),{"label":int(code),"label_confidence":int(conf.ravel()[flat]),"nlcd_cell_id":f"{transform.c + (flat%land.shape[1])*30:.2f}_{transform.f - (flat//land.shape[1])*30:.2f}"}))
    return ee.FeatureCollection(features),counts,records

# Use the official service pair advertised as the current C1.2 release.
# Stop if the provider changes its published release instead of reusing old assumptions.
release_page = requests.get("https://www.mrlc.gov/data", timeout=60)
release_page.raise_for_status()
if "Collection 1 Version 2 (1.2)" not in release_page.text:
    raise RuntimeError("USGS current release changed or could not be verified; review the official product metadata before training.")
release_dir = OUT / "official_nlcd" / "collection_1_2"
release_dir.mkdir(parents=True,exist_ok=True)
(release_dir / "release_page.html").write_text(release_page.text,encoding="utf-8")
source_meta = {"release": NLCD_SERVICE_RELEASE, "year": YEAR, "delivery": "Official MRLC raw WCS pair; native EPSG:5070 30 m grid", "release_page": release_page.url, "checked_utc": datetime.now(timezone.utc).isoformat(), "confidence_note": "Uncalibrated model confidence with documented caveats; high scores do not establish truth."}
USE_CONFIDENCE = True
print("Using official USGS 2022 labels and confidence; confidence filters enabled.")

# Audit target-label availability before expensive imagery sampling.
# Do not silently relax confidence filters if they remove a target class.
confidence_audit = []
for region_id, split, bounds in REGIONS:
    _, counts, records = official_training_points(bounds,region_id)
    source_meta.setdefault("regions",{})[region_id] = records
    confidence_audit.extend([{**row,"split":split} for row in counts])
confidence_audit = pd.DataFrame(confidence_audit)
confidence_audit.to_csv(OUT / "official_confidence_eligibility.csv",index=False)
(OUT / "official_nlcd_sources.json").write_text(json.dumps(source_meta,indent=2))
target_counts = confidence_audit.loc[confidence_audit["class"].isin([41,90])].groupby(["threshold","split","class"]).eligible_cells.sum()
display(target_counts.unstack("class",fill_value=0))
usable_thresholds = [t for t in CONFIDENCE_THRESHOLDS if all(target_counts.get((t,split,code),0) >= MIN_TRAIN_PER_CLASS for split in ["train","test"] for code in [41,90])]
if not usable_thresholds:
    raise ValueError("Official matched sources verified, but no configured confidence threshold provides enough forest AND wetland cells in both splits. See official_confidence_eligibility.csv. Expand/replace sampling regions or explicitly revise confidence criteria; thresholds have not been silently lowered. Existing prediction files belong to the previous run.")

requested_thresholds = list(CONFIDENCE_THRESHOLDS)
skipped_thresholds = sorted(set(requested_thresholds) - set(usable_thresholds))
if skipped_thresholds:
    warnings.warn(f"Skipping confidence thresholds with insufficient target coverage: {skipped_thresholds}. Using {usable_thresholds}; no thresholds silently lowered.")
CONFIDENCE_THRESHOLDS = usable_thresholds
source_meta["requested_thresholds"] = requested_thresholds
source_meta["active_thresholds"] = CONFIDENCE_THRESHOLDS
source_meta["skipped_thresholds"] = skipped_thresholds


## Build predictors and audit coverage
NAIP values remain digital numbers. Normalize them consistently by 255; these are not surface reflectance. Sentinel-2 values are scaled to reflectance before EVI. Texture is a 5x5 NAIP-pixel neighborhood. Sentinel-2 and terrain retain their coarser information even when predicted on a 2 m output grid.

The base model includes imagery, seasonal moisture and terrain. Absolute elevation is excluded to reduce geographic shortcuts. Relative elevation uses a 1 km neighborhood, not a hydraulic model or height-above-drainage product. SSURGO is optional and is never replaced by invented values.


In [ ]:
def clear_s2(image):
    scl = image.select("SCL")
    clear = scl.remap([0, 1, 3, 8, 9, 10, 11], [0]*7, 1).And(scl.mask())
    return image.updateMask(clear).select(["B2", "B3", "B4", "B8", "B11", "B12"]).multiply(0.0001)

def predictors(region):
    naip_col = ee.ImageCollection("USDA/NAIP/DOQQ").filterBounds(region).filterDate(f"{YEAR}-01-01", f"{YEAR+1}-01-01").filter(ee.Filter.listContains("system:band_names", "N"))
    if naip_col.size().getInfo() == 0:
        raise ValueError("No four-band NAIP for this region/year")
    first = ee.Image(naip_col.first())
    aerial = naip_col.sort("system:time_start").mosaic().setDefaultProjection(first.select("R").projection())
    rgbn = aerial.select(["R", "G", "B", "N"]).divide(255).rename(["red", "green", "blue", "nir"])
    texture = aerial.select("N").toUint8().glcmTexture(size=2).select(["N_contrast", "N_ent", "N_idm"])
    stack = rgbn.addBands(texture).addBands(rgbn.normalizedDifference(["nir", "red"]).rename("naip_ndvi"))
    seasonal_ids = {}
    for name, start, end in [("winter", f"{YEAR}-01-01", f"{YEAR}-04-01"), ("summer", f"{YEAR}-06-01", f"{YEAR}-09-01")]:
        collection = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(region).filterDate(start, end)
        if collection.size().getInfo() == 0:
            raise ValueError(f"No {name} Sentinel-2 imagery")
        image = collection.map(clear_s2).median()
        evi = image.expression("2.5*(n-r)/(n+6*r-7.5*b+1)", {"n": image.select("B8"), "r": image.select("B4"), "b": image.select("B2")}).rename(f"{name}_evi")
        ndmi = image.normalizedDifference(["B8", "B11"]).rename(f"{name}_ndmi")
        swir = image.select(["B11", "B12"], [f"{name}_swir1", f"{name}_swir2"])
        stack = stack.addBands(evi).addBands(ndmi).addBands(swir)
        seasonal_ids[name] = collection.aggregate_array("system:index").getInfo()
    dem_col = ee.ImageCollection("USGS/3DEP/10m_collection").filterBounds(region.buffer(2000))
    # Pin neighborhood calculations to the terrain grid. A 1 km radius at 2 m
    # would require a 1001-pixel kernel, exceeding Earth Engine's limit.
    terrain_projection = ee.Projection(METRIC_CRS).atScale(10)
    dem = dem_col.mosaic().select(0).reproject(terrain_projection)
    neighborhood_mean = dem.focalMean(radius=1000, units="meters").reproject(terrain_projection)
    relative = dem.subtract(neighborhood_mean).rename("relative_elevation").reproject(terrain_projection)
    terrain_slope = ee.Terrain.slope(dem).rename("slope").reproject(terrain_projection)
    stack = stack.addBands(relative).addBands(terrain_slope)
    if SOIL_PATH is not None:
        soil = gpd.read_file(SOIL_PATH).to_crs(4326)
        required = ["hydric_pct", "drainage_code"]
        if not all(k in soil for k in required):
            raise ValueError("SSURGO prepared polygons need hydric_pct and drainage_code")
        if soil[required].isna().any().any():
            raise ValueError("Resolve missing soil attributes before using soils")
        fc = ee.FeatureCollection(json.loads(soil[required + ["geometry"]].to_json()))
        for field in required:
            stack = stack.addBands(fc.reduceToImage([field], ee.Reducer.first()).rename(field))
    return stack.toFloat(), {"naip_ids": naip_col.aggregate_array("system:index").getInfo(), "sentinel_ids": seasonal_ids}


## Sample once per native NLCD cell
Labels and confidence now come together from official MRLC Collection 1.2 services. Both are downloaded at native 30 m resolution on the same grid. The previous community-mirror fallback is removed.

The point is the center of a 30 m label cell; predictors are sampled at that location at 2 m. Nearby subpixels are not duplicated as independent labels. All points from a region stay in one split. Goat Island and a 10 km buffer are excluded. Confidence-80 holdouts are fixed for both training thresholds. Samples are drawn separately from confidence bins 80–89 and 90–100 so high-confidence cases are represented.

Training-only quota: deciduous forest requests 240 samples per confidence bin and region, versus 120 for other classes. Original samples are retained and new unique native cells are added. Holdout quotas and random draws are unchanged. Confidence remains at least 80; 90 remains subject to availability checks.


In [ ]:
frames, audit, imagery_provenance = [], [], {}
feature_names = None
for region_id, split, bounds in REGIONS:
    print(f"Sampling {region_id} ({split})", flush=True)
    region = ee.Geometry.Rectangle(bounds, geodesic=False).difference(exclusion, 1)
    if region.area(1).getInfo() < 10000:
        raise ValueError(f"Region {region_id} falls inside study exclusion")
    stack, imagery_provenance[region_id] = predictors(region)
    names = stack.bandNames().getInfo()
    if feature_names is not None and names != feature_names:
        raise ValueError("Feature schema differs between regions")
    feature_names = names
    points, eligibility, records = official_training_points(bounds, region_id)
    eligibility_path = OUT / f"{region_id}_confidence_eligibility.csv"
    pd.DataFrame(eligibility).to_csv(eligibility_path,index=False)
    source_meta.setdefault("regions",{})[region_id] = records
    sampled = stack.sampleRegions(collection=points, properties=["label", "label_confidence", "nlcd_cell_id"], scale=OUTPUT_SCALE_M, geometries=True, tileScale=4).getInfo()
    rows = []
    for f in sampled["features"]:
        row = f["properties"]
        row.update(region=region_id, split=split, longitude=f["geometry"]["coordinates"][0], latitude=f["geometry"]["coordinates"][1])
        rows.append(row)
    if not rows:
        raise ValueError(f"No eligible samples in {region_id}; inspect its confidence_eligibility.csv before changing thresholds")
    frame = pd.DataFrame(rows)
    print(frame.label.value_counts().to_dict(), flush=True)
    frames.append(frame)
    audit.append({"region": region_id, "split": split, "samples": len(frame), "counts": frame.label.value_counts().to_dict()})
samples = pd.concat(frames, ignore_index=True)
if samples.groupby("nlcd_cell_id").split.nunique().gt(1).any():
    raise ValueError("One native label cell appears in both train and holdout; adjust region boundaries")
samples = samples.drop_duplicates("nlcd_cell_id").reset_index(drop=True)
if not np.isfinite(samples[feature_names].to_numpy()).all():
    raise ValueError("Non-finite predictors found")
samples.to_csv(OUT / "training_and_holdout_samples.csv", index=False)
gpd.GeoDataFrame(samples, geometry=gpd.points_from_xy(samples.longitude, samples.latitude), crs=4326).to_file(OUT / "samples.geojson", driver="GeoJSON")
(OUT / "sampling_audit.json").write_text(json.dumps(audit, indent=2), encoding="utf-8")
display(samples.groupby(["split", "label"]).size().unstack(fill_value=0))


## Fit models and compare confidence thresholds
Fit NAIP-only and full-feature models for thresholds 80 and 90. Class-count checks remain active; a threshold with inadequate target-class examples is not forced to run. Report metrics on fixed, spatially separate NLCD-labeled holdouts. This is weak-label agreement, not independent 2 m accuracy. The pilot chooses the highest holdout macro F1; this holdout is used for model selection, so it is not an untouched final test. Reserve additional regions for any final performance claim.


In [ ]:
train = samples.loc[samples.split.eq("train")].copy()
test = samples.loc[samples.split.eq("test")].copy()
train_counts, test_counts = train.label.value_counts(), test.label.value_counts()
active_classes = sorted(c for c in set(train.label) & set(test.label) if train_counts[c] >= MIN_TRAIN_PER_CLASS and test_counts[c] >= MIN_TRAIN_PER_CLASS)
omitted_classes = sorted((set(train.label) | set(test.label)) - set(active_classes))
if omitted_classes:
    warnings.warn(f"Classes with insufficient train/holdout examples excluded from this pilot: {omitted_classes}. Their true pixels cannot be evaluated by this model; predictions may assign an included class.")
samples.groupby(["split", "label"]).size().rename("count").to_csv(OUT / "class_counts.csv")
train = train.loc[train.label.isin(active_classes)].copy()
test = test.loc[test.label.isin(active_classes)].copy()
if not {41, 90}.issubset(set(train.label)) or not {41, 90}.issubset(set(test.label)):
    raise ValueError("Insufficient forest (41) or woody wetland (90) examples. Inspect class_counts.csv. Rerun settings, predictor definitions, and sampling with the updated regions; do not reuse old samples.")
feature_sets = {"naip_only": [c for c in feature_names if not c.startswith(("winter_", "summer_")) and c not in ["relative_elevation", "slope", "hydric_pct", "drainage_code"]], "full": feature_names}
models, metrics = {}, []
for threshold in (CONFIDENCE_THRESHOLDS if USE_CONFIDENCE else [None]):
    subset = train.loc[train.label_confidence.ge(threshold)] if threshold is not None else train
    counts = subset.label.value_counts()
    if any(counts.get(c, 0) < MIN_TRAIN_PER_CLASS for c in active_classes):
        warnings.warn(f"Skipping {threshold}: inadequate examples for one or more classes; expand sampling")
        continue
    for feature_set, columns in feature_sets.items():
        key = f"{feature_set}_confidence{threshold}" if threshold is not None else f"{feature_set}_patch_interior_only"
        model = RandomForestClassifier(n_estimators=300, min_samples_leaf=3, max_features="sqrt", class_weight="balanced_subsample", n_jobs=-1, random_state=SEED)
        model.fit(subset[columns], subset.label)
        prediction = model.predict(test[columns])
        report = classification_report(test.label, prediction, labels=active_classes, target_names=[CLASSES[c] for c in active_classes], output_dict=True, zero_division=0)
        pd.DataFrame(report).T.to_csv(OUT / f"{key}_class_report.csv")
        pd.DataFrame(confusion_matrix(test.label, prediction, labels=active_classes), index=active_classes, columns=active_classes).to_csv(OUT / f"{key}_confusion_matrix.csv")
        models[key] = (model, columns)
        metrics.append({"model": key, "training_rows": len(subset), "holdout_rows": len(test), "macro_f1": f1_score(test.label, prediction, labels=active_classes, average="macro", zero_division=0)})
if not models:
    raise ValueError("No model passed sample-count checks; expand training regions or reduce class scope explicitly")
metrics = pd.DataFrame(metrics).sort_values("macro_f1", ascending=False)
metrics.to_csv(OUT / "model_comparison.csv", index=False)
BEST_MODEL = metrics.iloc[0].model
model, columns = models[BEST_MODEL]
joblib.dump({"model": model, "features": columns, "classes": CLASSES, "year": YEAR}, OUT / "selected_model.joblib")
pd.Series(model.feature_importances_, index=columns).sort_values(ascending=False).to_csv(OUT / "feature_importance.csv", header=["importance"])
display(metrics)

region_metrics = []
for region_id, frame in test.groupby("region"):
    predicted = model.predict(frame[columns])
    report = classification_report(frame.label, predicted, labels=active_classes, output_dict=True, zero_division=0)
    region_metrics.append({"region":region_id,"rows":len(frame),"macro_f1_all_active_classes":f1_score(frame.label,predicted,labels=active_classes,average="macro",zero_division=0),"forest_reference_rows":int(frame.label.eq(41).sum()),"wetland_reference_rows":int(frame.label.eq(90).sum()),"forest_f1":report.get("41",{}).get("f1-score"),"wetland_f1":report.get("90",{}).get("f1-score")})
pd.DataFrame(region_metrics).to_csv(OUT / "geographic_holdout_metrics.csv",index=False)
display(pd.DataFrame(region_metrics))


## Export Goat Island predictors and predict locally
No NLCD label or confidence is included as a predictor. Large exports can take time; failed downloads stop the notebook. Model class probabilities are uncalibrated vote-based scores, not probabilities of ecological truth. Low-confidence locations remain visible in the confidence map.


In [ ]:
# Small fixed-grid requests stay below Earth Engine's per-download limit.
# Downloads are cached by image expression and grid, so interrupted runs can resume.
import math
import time
import requests
from contextlib import ExitStack
from rasterio.merge import merge

def download_predictors_tiled(image, destination, band_names, tile_pixels=512):
    scale = OUTPUT_SCALE_M
    xmin, ymin, xmax, ymax = preserve_m.total_bounds
    left, top = math.floor(xmin / scale) * scale, math.ceil(ymax / scale) * scale
    width = math.ceil((xmax - left) / scale)
    height = math.ceil((top - ymin) / scale)
    tile_image = image.select(band_names).toFloat().unmask(-9999, False)
    signature = hashlib.sha256((tile_image.serialize() + json.dumps([left, top, width, height, scale, band_names])).encode()).hexdigest()[:16]
    tile_dir = OUT / "predictor_tiles" / signature
    tile_dir.mkdir(parents=True, exist_ok=True)
    tile_files = []
    for row in range(0, height, tile_pixels):
        for col in range(0, width, tile_pixels):
            w, h = min(tile_pixels, width-col), min(tile_pixels, height-row)
            transform = rasterio.Affine(scale, 0, left+col*scale, 0, -scale, top-row*scale)
            path = tile_dir / f"tile_{row}_{col}.tif"
            cached = False
            if path.exists():
                try:
                    with rasterio.open(path) as src:
                        cached = src.count == len(band_names) and src.shape == (h,w) and src.transform.almost_equals(transform) and src.crs == rasterio.crs.CRS.from_string(METRIC_CRS)
                except rasterio.errors.RasterioIOError:
                    cached = False
            if not cached:
                for attempt in range(3):
                    try:
                        params = {"crs": METRIC_CRS, "crs_transform": list(transform)[:6], "dimensions": [w,h], "format": "GEO_TIFF"}
                        url = tile_image.getDownloadURL(params)
                        response = requests.get(url, timeout=300)
                        response.raise_for_status()
                        partial = path.with_suffix(".part.tif")
                        partial.write_bytes(response.content)
                        with rasterio.open(partial) as src:
                            if src.count != len(band_names) or src.shape != (h,w) or not src.transform.almost_equals(transform):
                                raise ValueError("Downloaded tile grid or band count mismatch")
                        partial.replace(path)
                        break
                    except Exception:
                        if attempt == 2:
                            raise
                        time.sleep(2 * (attempt+1))
            tile_files.append(path)
            print(f"Predictor tile {len(tile_files)}: {'cached' if cached else 'downloaded'}", flush=True)
    with ExitStack() as context:
        sources = [context.enter_context(rasterio.open(path)) for path in tile_files]
        mosaic, transform = merge(sources, nodata=-9999)
        profile = sources[0].profile.copy()
        profile.update(height=height, width=width, transform=transform, count=len(band_names), dtype="float32", nodata=-9999, compress="deflate")
        if mosaic.shape != (len(band_names), height, width):
            raise ValueError("Merged predictor dimensions differ from expected grid")
        temporary = destination.with_suffix(".partial.tif")
        with rasterio.open(temporary,"w",**profile) as dst:
            dst.write(mosaic)
            dst.descriptions = tuple(band_names)
        temporary.replace(destination)
    (tile_dir / "grid.json").write_text(json.dumps({"bands":band_names,"scale_m":scale,"width":width,"height":height,"tile_pixels":tile_pixels,"signature":signature},indent=2))
    return destination

goat_stack, imagery_provenance["goat_island"] = predictors(roi)
feature_path = OUT / f"goat_predictors_{YEAR}_{OUTPUT_SCALE_M}m.tif"
download_predictors_tiled(goat_stack.clip(roi), feature_path, columns)
with rasterio.open(feature_path) as src:
    profile = src.profile.copy()
    array = src.read(masked=True)
    inside = geometry_mask([preserve_m.geometry.iloc[0].__geo_interface__], out_shape=src.shape, transform=src.transform, invert=True)
    valid = inside & ~np.ma.getmaskarray(array).any(axis=0) & np.isfinite(array.data).all(axis=0)
    if not valid.any():
        raise ValueError("No common valid predictor pixels at Goat Island")
    classified = np.full(src.shape, 255, dtype="uint8")
    probability = np.full(src.shape, -9999, dtype="float32")
    flat = np.flatnonzero(valid)
    for offset in range(0, len(flat), 100000):
        loc = flat[offset:offset+100000]
        X = pd.DataFrame(array.data.reshape(len(columns), -1)[:, loc].T, columns=columns)
        probs = model.predict_proba(X)
        classified.ravel()[loc] = model.classes_[probs.argmax(axis=1)]
        probability.ravel()[loc] = probs.max(axis=1)
    for filename, values, dtype, nodata in [("land_cover.tif", classified, "uint8", 255), ("model_confidence.tif", probability, "float32", -9999)]:
        profile.update(count=1, dtype=dtype, nodata=nodata)
        with rasterio.open(OUT / filename, "w", **profile) as dst:
            dst.write(values, 1)
    area = abs(src.transform.a * src.transform.e)
    coverage = {"valid_cells": int(valid.sum()), "parcel_cells": int(inside.sum()), "valid_pct": float(100*valid.sum()/inside.sum())}
    pd.DataFrame([{"code": c, "class": CLASSES[int(c)], "raster_acres": float((classified == c).sum()*area/4046.8564224)} for c in model.classes_]).to_csv(OUT / "class_areas.csv", index=False)
    (OUT / "prediction_coverage.json").write_text(json.dumps(coverage, indent=2))
print("Prediction coverage:", coverage)


In [ ]:
palette = plt.get_cmap("tab20")
class_codes = list(CLASSES)
colors = [palette(i / len(class_codes)) for i in range(len(class_codes))]
with rasterio.open(OUT / "land_cover.tif") as src:
    values = src.read(1)
    indexed = np.full(src.shape, np.nan)
    for i, c in enumerate(class_codes):
        indexed[values == c] = i
    fig, ax = plt.subplots(figsize=(12, 7))
    show(np.ma.masked_invalid(indexed), transform=src.transform, ax=ax, cmap=ListedColormap(colors), norm=BoundaryNorm(np.arange(len(class_codes)+1)-0.5, len(class_codes)), interpolation="nearest")
    preserve_m.boundary.plot(ax=ax, color="black", linewidth=0.8)
    ax.set_title(f"Goat Island experimental land cover — {YEAR}")
    ax.set_axis_off()
    ax.legend(handles=[Patch(color=colors[i], label=CLASSES[c]) for i,c in enumerate(class_codes) if (values == c).any()], bbox_to_anchor=(1.01,1), loc="upper left")
    fig.text(.5,.02,"NLCD-trained predictions • 2 m output is not independently validated 2 m accuracy",ha="center",fontsize=8)
    fig.savefig(OUT / "land_cover.png", dpi=200, bbox_inches="tight")
    plt.show()
with rasterio.open(OUT / "model_confidence.tif") as src:
    fig, ax = plt.subplots(figsize=(9, 7))
    show(src.read(1, masked=True), transform=src.transform, ax=ax, cmap="viridis", vmin=0, vmax=1)
    fig.colorbar(ax.images[0], ax=ax, label="Maximum RF class score (uncalibrated)")
    ax.set(title="Goat Island prediction confidence")
    ax.set_axis_off()
    fig.savefig(OUT / "model_confidence.png", dpi=200, bbox_inches="tight")
    plt.show()


## External NWI comparison (optional)
Download NWI from https://www.fws.gov/program/national-wetlands-inventory/download-state-wetlands-data and set NWI_PATH above. Record the source imagery date and inventory metadata. Forested wetlands use Cowardin PFO codes, which are not an exact equivalent of NLCD woody wetlands (which also include shrubs). Compare separately with all vegetated wetlands. Absence from NWI is not proof of non-wetland. These are overlap statistics, not independent classification accuracy.


In [ ]:
if NWI_PATH is None:
    print("NWI comparison not run: supply a downloaded polygon file in NWI_PATH. NWI has not entered training.")
else:
    from rasterio.features import rasterize
    nwi = gpd.read_file(NWI_PATH).to_crs(METRIC_CRS)
    if "ATTRIBUTE" not in nwi:
        raise ValueError("Expected NWI ATTRIBUTE field")
    rows = []
    with rasterio.open(OUT / "land_cover.tif") as src:
        pred = src.read(1)
        for name, pattern in [("forested_wetland", "PFO"), ("vegetated_wetland", "PFO|PSS|PEM")]:
            subset = nwi.loc[nwi.ATTRIBUTE.str.contains(pattern, regex=True, na=False)]
            subset = gpd.clip(subset, preserve_m)
            mask = rasterize([(g,1) for g in subset.geometry if not g.is_empty], out_shape=src.shape, transform=src.transform, fill=0) if len(subset) else np.zeros(src.shape)
            mapped = mask.astype(bool) & (pred != 255)
            rows.append({"nwi_group": name, "nwi_valid_pixels": int(mapped.sum()), "predicted_woody_wetland_within_nwi_pixels": int(((pred == 90) & mapped).sum()), "note": "External inventory overlap; not field accuracy"})
    pd.DataFrame(rows).to_csv(OUT / "nwi_comparison.csv", index=False)
    display(pd.DataFrame(rows))


In [ ]:
manifest = {"run_utc": datetime.now(timezone.utc).isoformat(), "year": YEAR, "boundary_hash": BOUNDARY_HASH, "parcel_ids": PARCEL_IDS, "training_regions": REGIONS, "training_class_quotas_per_confidence_bin": TRAIN_CLASS_QUOTAS, "thresholds": CONFIDENCE_THRESHOLDS if USE_CONFIDENCE else [], "confidence_filter_used": USE_CONFIDENCE, "omitted_classes": omitted_classes, "source_assets": {"labels": "https://dmsdata.cr.usgs.gov/geoserver/mrlc_Land-Cover-Native_conus_year_data/wcs", "confidence": "https://dmsdata.cr.usgs.gov/geoserver/mrlc_Land-Cover-Confidence-Native_conus_year_data/wcs"}, "nlcd_metadata": source_meta, "imagery_provenance": imagery_provenance, "features": columns, "selected_model": BEST_MODEL, "output_scale_m": OUTPUT_SCALE_M, "soil_path": str(SOIL_PATH), "nwi_path": str(NWI_PATH), "limitations": ["NLCD weak labels", "Confidence product has documented caveats", "Holdout used for selection, not an untouched final test", "NWI comparison is not field validation", "Coarse auxiliary data do not become 2 m information", "Existing constraints model not modified"]}
(OUT / "run_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("Saved experimental products:", OUT)


## Sources and next steps
- NAIP: https://developers.google.com/earth-engine/datasets/catalog/USDA_NAIP_DOQQ
- Sentinel-2: https://developers.google.com/earth-engine/datasets/catalog/COPERNICUS_S2_SR_HARMONIZED
- Annual NLCD mirror: https://gee-community-catalog.org/projects/annual_nlcd/
- Confidence caveats: https://www.usgs.gov/centers/eros/science/annual-nlcd-land-cover-confidence
- SSURGO: https://www.nrcs.usda.gov/resources/data-and-reports/soil-survey-geographic-database-ssurgo
- Terrain: https://www.usgs.gov/the-national-map-data-delivery/gis-data-download

Before expansion: inspect source versions and sample counts; inspect woody-wetland versus forest confusion; prepare soils if needed; obtain NWI; reserve additional regions for final evaluation. Multi-year change detection is a later experiment after single-year assessment, and is not claimed by this notebook.
